[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C57_Small_Object_Detection_Course/03_assignment_loss/03_assignment_loss_small.ipynb)

# 03 · 分配与损失层面的解法（NWD / RFLA / center-based / IoU 变体 / scale-aware 加权）

目标：把「IoU 这把尺子对小框是坏的」从一句话变成一组**可以 assert 的等式**，
然后从零实现 **NWD**，并证明它恰好补上了 IoU 的三个短板。

**你会亲手实现：**
1. IoU 的位移灵敏度 `4/w` 与 NWD 的 `√2/C`——**一个反比于框长，一个是常数**
2. **NWD 全套**：bbox → 2D 高斯 → 2-Wasserstein 闭式解 → 指数归一化（三行）
3. **关键对比实验**：正样本容忍半径、死区比例、跨尺度一致性，逐条 assert
4. stride-8 网格上的**匹配率**：IoU@0.5 下 8 px 目标只有 ~25%，NWD@0.5 下 100%
5. center-based 分配 + **从设计目标反解出来的**尺度自适应阈值 `τ(w) = (w−r₀)/(w+r₀)`
6. ATSS 的自适应阈值、SimOTA 的 dynamic-k（**证明它对小目标方向是反的**）
7. RFLA 的 KLD——并算清楚**方向搞反会比 IoU 更不公平**
8. IoU / GIoU / DIoU / CIoU / EIoU / SIoU 全套，验证 **GIoU 奖励放大、NWD 惩罚放大**
9. scale-aware 加权的**内点最优 γ**，以及噪声方差被放大 380 倍

> 心智模型：**小目标面对的威胁（网格量化 ±4 px、标注抖动 ±2 px）都是绝对像素量，
> 而 IoU 是相对度量。用相对度量去衡量绝对误差，就是整个问题的根源。
> NWD 的全部价值，就是把位置项换成绝对距离。**

## 1 · 把问题钉死：IoU 的位移灵敏度是 4/w

In [ ]:
import numpy as np, math
rng = np.random.default_rng(573)

def to_xyxy(b):
    b = np.asarray(b, dtype=float)
    cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return np.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], axis=-1)

def iou(a, b):
    '''a, b: (..., 4) 的 (cx, cy, w, h)。支持广播。'''
    A, B = to_xyxy(a), to_xyxy(b)
    x1 = np.maximum(A[..., 0], B[..., 0]); y1 = np.maximum(A[..., 1], B[..., 1])
    x2 = np.minimum(A[..., 2], B[..., 2]); y2 = np.minimum(A[..., 3], B[..., 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar_a = np.clip(A[...,2]-A[...,0], 0, None) * np.clip(A[...,3]-A[...,1], 0, None)
    ar_b = np.clip(B[...,2]-B[...,0], 0, None) * np.clip(B[...,3]-B[...,1], 0, None)
    return inter / np.maximum(ar_a + ar_b - inter, 1e-12)

# 模块 01 的两个招牌数字，先对上
assert abs(float(iou((0,0,8,8),  (2,2,8,8)))  - 36/92)     < 1e-12
assert abs(float(iou((0,0,64,64),(2,2,64,64))) - 3844/4348) < 1e-12
print(f'8x8  框对角位移 2px: IoU = {float(iou((0,0,8,8),(2,2,8,8))):.4f}')
print(f'64x64 框对角位移 2px: IoU = {float(iou((0,0,64,64),(2,2,64,64))):.4f}')
print(f'=> 同样 2 px 的绝对误差，小框被惩罚了 '
      f'{float(iou((0,0,64,64),(2,2,64,64)))/float(iou((0,0,8,8),(2,2,8,8))):.2f} 倍')

## 2 · 从零实现 NWD：bbox → 2D 高斯 → W₂ → exp 归一化

In [ ]:
C_NWD = 12.8          # NWD 唯一的超参：取数据集目标 sqrt(wh) 的均值（AI-TOD 用 12.8）

def box_to_gauss(b):
    '''(cx,cy,w,h) -> (mu, Sigma^{1/2} 的对角元)。
       Sigma = diag(w²/4, h²/4)  =>  Sigma^{1/2} = diag(w/2, h/2)。
       物理含义：把框的**内切椭圆**当作高斯的等密度线。'''
    b = np.asarray(b, dtype=float)
    return b[..., :2], b[..., 2:] / 2.0

def w2_sq(a, b):
    '''两个对角高斯的 2-Wasserstein 距离平方（闭式解）：
       W2² = ||mu_a - mu_b||² + ||Sigma_a^{1/2} - Sigma_b^{1/2}||_F²
       对角情形下 = 把框写成 [cx, cy, w/2, h/2] 之后的欧氏距离平方。'''
    mu_a, s_a = box_to_gauss(a)
    mu_b, s_b = box_to_gauss(b)
    return ((mu_a - mu_b) ** 2).sum(-1) + ((s_a - s_b) ** 2).sum(-1)

def nwd(a, b, C=C_NWD):
    '''NWD = exp(-sqrt(W2²)/C)，取值 (0, 1]，= 1 当且仅当两框完全相同。'''
    return np.exp(-np.sqrt(w2_sq(a, b)) / C)

# —— 正确性校验：手算对拍 ——
assert abs(float(w2_sq((0,0,8,8), (2,2,8,8))) - 8.0) < 1e-12       # 4+4+0+0
assert abs(float(w2_sq((0,0,8,8), (0,0,16,16))) - 2*16.0) < 1e-12  # 0+0+4²+4²
assert abs(float(nwd((3,7,11,5), (3,7,11,5))) - 1.0) < 1e-15,  'NWD(x,x) 必须恰为 1'
assert 0 < float(nwd((0,0,8,8), (999,999,8,8))) < 1e-40,        'NWD 恒 > 0（永不进死区）'
manual = math.exp(-math.sqrt(8.0) / C_NWD)
assert abs(float(nwd((0,0,8,8), (2,2,8,8))) - manual) < 1e-15
print(f'W2²((0,0,8,8),(2,2,8,8)) = {float(w2_sq((0,0,8,8),(2,2,8,8))):.1f}')
print(f'NWD = exp(-sqrt(8)/12.8) = {manual:.6f}')
print('✅ 三行实现，处处可导，且不需要求交集。')

In [ ]:
# **性质一（核心）**：相同绝对位移 -> 相同 NWD，与框大小无关（可精确 assert）
print(f"{'框边长':>8s} {'IoU(位移2px)':>14s} {'NWD(位移2px)':>14s}")
nwds = []
for w in [4, 8, 16, 32, 64, 128, 256]:
    i_ = float(iou((0,0,w,w), (2,2,w,w)))
    n_ = float(nwd((0,0,w,w), (2,2,w,w)))
    nwds.append(n_)
    print(f'{w:>8d} {i_:>14.4f} {n_:>14.6f}')

assert max(nwds) - min(nwds) < 1e-14, 'NWD 对同一绝对位移在所有尺度上完全相同'
ious = [float(iou((0,0,w,w), (2,2,w,w))) for w in [4, 8, 16, 32, 64, 128, 256]]
pen  = [1.0 - v for v in ious]          # 「惩罚」= 1 - IoU，差异看得更清楚
assert max(ious) / min(ious) > 6.0, 'IoU 本身在不同尺度上相差数倍'
assert max(pen) / min(pen) > 25.0,  '按「惩罚」看差了一个多量级'
print(f'\n⚠️  IoU 最大/最小 = {max(ious)/min(ious):.1f} 倍；'
      f'按惩罚(1−IoU) 看是 {max(pen)/min(pen):.0f} 倍   ❌ 极不公平')
print(f'✅ NWD: 最大−最小 = {max(nwds)-min(nwds):.2e}   —— 是恒等式，不是近似')

## 3 · 关键对比实验：灵敏度、死区、容忍半径

In [ ]:
# (a) 位移灵敏度： |d(metric)/d(delta)| 在 delta=0 处
def sens_at_zero(metric, w, eps=1e-3):
    '''沿对角线每轴位移 eps，用有限差分估计灵敏度。'''
    return (metric((0,0,w,w), (0,0,w,w)) - metric((0,0,w,w), (eps,eps,w,w))) / eps

print(f"{'框边长 w':>10s} {'IoU 灵敏度':>12s} {'理论 4/w':>10s} "
      f"{'NWD 灵敏度':>12s} {'理论 √2/C':>11s}")
for w in [4, 8, 16, 32, 64, 128]:
    si = float(sens_at_zero(lambda a,b: iou(a,b), w))
    sn = float(sens_at_zero(lambda a,b: nwd(a,b), w))
    assert abs(si - 4.0/w) < 0.01, (w, si)
    assert abs(sn - math.sqrt(2)/C_NWD) < 1e-4, (w, sn)
    print(f'{w:>10d} {si:>12.4f} {4.0/w:>10.4f} {sn:>12.4f} {math.sqrt(2)/C_NWD:>11.4f}')

s4, s128 = float(sens_at_zero(lambda a,b: iou(a,b), 4)), \
           float(sens_at_zero(lambda a,b: iou(a,b), 128))
assert abs(s4/s128 - 32.0) < 0.5
print(f'\n⚠️  IoU 的灵敏度 = 4/w —— 4px 框比 128px 框陡 {s4/s128:.0f} 倍')
print(f'✅ NWD 的灵敏度 = √2/C = {math.sqrt(2)/C_NWD:.4f} —— **与 w 完全无关**')

In [ ]:
# (b) 死区：IoU 在不相交时恒为 0（梯度也为 0）；NWD 永远 > 0
print(f"{'框边长':>8s} {'δ∈[0,2w] 上 IoU==0 的比例':>26s} {'NWD 最小值':>14s}")
for w in [8, 32, 128]:
    d = np.linspace(0, 2*w, 401)
    boxes_a = np.tile(np.array([0., 0., w, w]), (len(d), 1))
    boxes_b = np.stack([d, d, np.full(len(d), float(w)), np.full(len(d), float(w))], 1)
    iv, nv = iou(boxes_a, boxes_b), nwd(boxes_a, boxes_b)
    dead = float((iv <= 0).mean())
    assert dead > 0.49, (w, dead)
    assert nv.min() > 0
    print(f'{w:>8d} {dead:>25.1%} {nv.min():>14.3e}')
print('\n⚠️  IoU 在一半以上的位移范围内恒为 0：错 9px 和错 90px 在它眼里一模一样。')
print('✅ NWD 恒 > 0；但注意 C 决定了有效范围 —— d >> C 时它指数衰减，梯度同样消失。')
print('   经验规则：C ≈ 数据集目标 sqrt(wh) 的均值。')

In [ ]:
# (c) 正样本容忍半径（沿轴位移）：IoU 正比于 w，NWD 恒定
def iou_radius_axis(w, tau):
    '''解 (w-r)/(w+r) = tau  =>  r = w(1-tau)/(1+tau)'''
    return w * (1.0 - tau) / (1.0 + tau)

def nwd_radius(C, tau):
    '''解 exp(-r/C) = tau  =>  r = -C ln(tau)   （等尺寸框，Sigma 项抵消）'''
    return -C * math.log(tau)

# 数值验证两个闭式解
for w, tau in [(8, 0.5), (32, 0.3), (128, 0.7)]:
    r = iou_radius_axis(w, tau)
    assert abs(float(iou((0,0,w,w), (r,0,w,w))) - tau) < 1e-9
r = nwd_radius(C_NWD, 0.5)
assert abs(float(nwd((0,0,20,20), (r,0,20,20))) - 0.5) < 1e-9

GRID_ERR = 4.0        # stride 8 网格：GT 中心到最近格子中心的偏移，每轴最大 4 px
print(f"{'w':>6s} {'r(IoU@0.5)':>12s} {'r(IoU@0.3)':>12s} {'r(NWD@0.5)':>12s} {'够不够 4px 量化误差'}")
for w in [4, 8, 16, 64, 256]:
    r5, r3, rn = iou_radius_axis(w,0.5), iou_radius_axis(w,0.3), nwd_radius(C_NWD,0.5)
    ok5 = '✅' if r5 >= GRID_ERR else '❌'
    print(f'{w:>6d} {r5:>11.2f}{ok5} {r3:>12.2f} {rn:>12.2f} '
          f'{"IoU@0.5 覆盖不了网格误差" if r5 < GRID_ERR else ""}')

assert iou_radius_axis(8, 0.5) < GRID_ERR, '8px 框在 IoU@0.5 下的容忍半径小于网格量化误差'
assert iou_radius_axis(256, 0.3) > 100, '把阈值降到 0.3 会让大框的容忍半径失控'
print(f'\n⚠️  IoU@0.5: 8px 框只容忍 {iou_radius_axis(8,0.5):.2f}px < 网格固有误差 4px'
      f' -> **结构性不匹配**')
print(f'⚠️  降到 IoU@0.3: 8px 够了({iou_radius_axis(8,0.3):.2f}px)，'
      f'但 256px 框变成 {iou_radius_axis(256,0.3):.0f}px -> 收进大量明显错的框')
print(f'✅ NWD@0.5: 所有尺度都是 {nwd_radius(C_NWD,0.5):.2f}px —— 由 C 与 τ 直接设计')

In [ ]:
# (d) 决定性实验：stride-8 网格上的**匹配率**（anchor 尺寸完美匹配，已是最有利情形）
def match_rate(size, stride, metric, thr, n=200_000, anchor_size=None):
    '''GT 中心随机 -> 到最近格子中心的偏移每轴 ~ U(-stride/2, stride/2)。'''
    a_sz = float(anchor_size if anchor_size is not None else size)
    d = rng.uniform(-stride/2.0, stride/2.0, size=(n, 2))
    gt  = np.tile(np.array([0., 0., float(size), float(size)]), (n, 1))
    anc = np.stack([d[:,0], d[:,1], np.full(n, a_sz), np.full(n, a_sz)], axis=1)
    return float((metric(gt, anc) >= thr).mean())

def adaptive_iou_thr(size, r0=4.0, tau_min=0.15, tau_max=0.90):
    '''**轴向**设计：让「沿轴位移」的容忍半径恒为 r0。
       解 w(1-τ)/(1+τ) = r0  =>  τ = (w-r0)/(w+r0)'''
    return float(np.clip((size - r0) / (size + r0), tau_min, tau_max))

def adaptive_iou_thr_diag(size, r0=4.5, tau_min=0.05, tau_max=0.90):
    '''**对角**设计：让「两轴各偏 r0」这个最坏角点恰好达到阈值。
       IoU_diag(δ) = a/(2-a)，a = (1-δ/w)²  =>  τ = a/(2-a)'''
    a = max(0.0, 1.0 - r0 / float(size)) ** 2
    return float(np.clip(a / (2.0 - a), tau_min, tau_max))

print(f"{'GT 尺寸':>8s} {'IoU@0.5':>9s} {'IoU@0.3':>9s} {'自适应(轴向)':>13s} "
      f"{'自适应(对角)':>13s} {'NWD@0.5':>9s}")
rates = {}
for size in [4, 8, 16, 32, 64]:
    r1 = match_rate(size, 8, lambda a,b: iou(a,b), 0.5)
    r2 = match_rate(size, 8, lambda a,b: iou(a,b), 0.3)
    r3 = match_rate(size, 8, lambda a,b: iou(a,b), adaptive_iou_thr(size))
    r3d = match_rate(size, 8, lambda a,b: iou(a,b), adaptive_iou_thr_diag(size))
    r4 = match_rate(size, 8, lambda a,b: nwd(a,b), 0.5)
    rates[size] = (r1, r2, r3, r3d, r4)
    print(f'{size:>8d} {r1:>8.1%} {r2:>8.1%} {r3:>12.1%} {r3d:>12.1%} {r4:>8.1%}')

assert 0.20 < rates[8][0]  < 0.30, rates[8][0]     # 8px + IoU@0.5 -> ~25%
assert 0.80 < rates[16][0] < 0.90, rates[16][0]    # 16px -> ~85%
assert rates[32][0] > 0.999                        # 32px -> 100%
assert rates[8][4] > 0.999 and rates[4][4] > 0.999, 'NWD@0.5 在所有尺度上都是 100%'
assert rates[8][3] > 0.999 and rates[8][3] > rates[8][2], '对角设计才真的拉到 100%'
print(f'\n⚠️  **即使 anchor 尺寸完美匹配**，8px 目标在 IoU@0.5 下也只有 '
      f'{rates[8][0]:.0%} 能匹配上 —— 另外 {1-rates[8][0]:.0%} 一个正样本都没有。')
print('    真实配置里 anchor 尺寸未必匹配，只会更差。')
print(f'⚠️  轴向设计的自适应阈值只能到 {rates[8][2]:.0%} —— 因为 **IoU 的等值面是各向异性的**：')
print('    同样 4px 的偏移，沿对角(4,4) 比沿轴(5.66,0) 更伤 IoU。')
print(f'✅ 按对角最坏角点设计后达到 {rates[8][3]:.0%}；NWD@0.5 直接就是 {rates[8][4]:.0%} ——')
print('    因为 **NWD 只依赖 ||Δμ||，等值面是标准圆**，根本没有各向异性这回事。')

# NWD 并没有退化成「什么都匹配」：尺寸差太多照样拒绝
n_mismatch = float(nwd((0,0,8,8), (0,0,32,32)))
assert n_mismatch < 0.5, n_mismatch
print(f'✅ 但 NWD 不是万能通行证：8px GT vs 32px anchor（中心重合）NWD = '
      f'{n_mismatch:.3f} < 0.5，照样拒绝 —— Σ 项显式惩罚尺寸不匹配。')

## 4 · center-based 分配与尺度自适应阈值

In [ ]:
def center_sampling(gt, points, radius_cells, stride):
    '''FCOS 式 center sampling：点必须同时
       ① 落在 GT 框内   ② 距 GT 中心 < radius_cells * stride
       对小目标 ② 的半径可能大于目标本身，所以**必须求交**，否则会收进背景点。'''
    gt = np.asarray(gt, float); pts = np.asarray(points, float)
    x1, y1, x2, y2 = gt[0]-gt[2]/2, gt[1]-gt[3]/2, gt[0]+gt[2]/2, gt[1]+gt[3]/2
    in_box = (pts[:,0] >= x1) & (pts[:,0] <= x2) & (pts[:,1] >= y1) & (pts[:,1] <= y2)
    r = radius_cells * stride
    in_rad = (np.abs(pts[:,0]-gt[0]) < r) & (np.abs(pts[:,1]-gt[1]) < r)
    return in_box & in_rad

STRIDE, R_CELLS = 8, 2.5          # YOLOX 的 center sampling 半径是 2.5 个格子
grid = np.array([[x + STRIDE/2, y + STRIDE/2]
                 for x in range(0, 128, STRIDE) for y in range(0, 128, STRIDE)])
print(f"{'GT 尺寸':>8s} {'只用半径':>10s} {'只在框内':>10s} {'两者求交':>10s}")
n_rad_only = None
for size in [6, 8, 16, 32, 64]:
    g = (64., 64., float(size), float(size))
    only_rad = int(((np.abs(grid[:,0]-64) < R_CELLS*STRIDE) &
                    (np.abs(grid[:,1]-64) < R_CELLS*STRIDE)).sum())
    n_rad_only = only_rad
    x1,y1,x2,y2 = 64-size/2, 64-size/2, 64+size/2, 64+size/2
    only_box = int(((grid[:,0]>=x1)&(grid[:,0]<=x2)&(grid[:,1]>=y1)&(grid[:,1]<=y2)).sum())
    both = int(center_sampling(g, grid, R_CELLS, STRIDE).sum())
    print(f'{size:>8d} {only_rad:>10d} {only_box:>10d} {both:>10d}')

n6  = int(center_sampling((64.,64.,6.,6.),   grid, R_CELLS, STRIDE).sum())
n64 = int(center_sampling((64.,64.,64.,64.), grid, R_CELLS, STRIDE).sum())
assert n6 <= 1 and n_rad_only >= 9 and n64 >= 9, (n6, n_rad_only, n64)
print(f'\n⚠️  只用半径 -> 6px 的目标会拿到 {n_rad_only} 个正样本，'
      f'而它们的特征**几乎全是背景**（目标只有 6px，半径覆盖 40px）。')
print(f'✅ 必须与「点在框内」求交。但求交之后，6px 目标只剩 {n6} 个正样本 ——')
print('   这个上限只有更高分辨率（P2）能突破，分配规则本身解决不了。')

In [ ]:
# 尺度自适应阈值：从设计目标（容忍半径恒为 r0）反解，而不是拍脑袋定表
print(f"{'w':>6s} {'τ(w), r0=4':>12s} {'反算半径':>10s} {'固定τ=0.5 的半径':>18s}")
for w in [8, 16, 32, 64, 256]:
    t = adaptive_iou_thr(w, r0=4.0)
    r = iou_radius_axis(w, t)
    print(f'{w:>6d} {t:>12.3f} {r:>10.2f} {iou_radius_axis(w,0.5):>18.2f}')

for w in [8, 16, 32, 64]:
    assert abs(iou_radius_axis(w, adaptive_iou_thr(w, r0=4.0)) - 4.0) < 1e-9, w
assert adaptive_iou_thr(256, r0=4.0) == 0.90, '大框会被 tau_max 截断（有意为之）'
print('\n✅ τ(w) = (w−r0)/(w+r0) 让容忍半径在所有尺度上恒为 r0。')
print('   r0 的取法：网格量化误差(stride/2) + 标注抖动(±1~2px) + 亚像素回归精度。')
print('   ⚠️  注意：这与「固定阈值的 NWD」在做同一件事 —— 让容忍半径与尺度解耦。')
print('       区别是它只补偿了「判正负」这一个点，回归损失里的 IoU 该多陡还是多陡。')

## 5 · ATSS 的自适应阈值，与 SimOTA dynamic-k 的反向作用

In [ ]:
def atss_threshold(gt, cand_boxes, metric):
    '''ATSS: 阈值 = 候选集上度量值的 均值 + 标准差。'''
    v = metric(np.tile(np.asarray(gt, float), (len(cand_boxes), 1)),
               np.asarray(cand_boxes, float))
    return float(v.mean() + v.std()), v

STRIDES = (4, 8, 16, 32)
# 每层的 anchor：RetinaNet 惯例 base scale = 4 x stride
LEVEL_ANCHORS = {}
for st in STRIDES:
    LEVEL_ANCHORS[st] = np.array([[x + st/2, y + st/2, 4.0*st, 4.0*st]
                                  for x in range(0, 256, st) for y in range(0, 256, st)])
anchors = np.concatenate([LEVEL_ANCHORS[st] for st in STRIDES], axis=0)

def topk_per_level(gt, k=9):
    '''ATSS 的候选集：**每层各取 k 个中心最近的 anchor**，再并起来。'''
    out = []
    for st in STRIDES:
        A = LEVEL_ANCHORS[st]
        d = np.linalg.norm(A[:, :2] - np.asarray(gt, float)[:2], axis=1)
        out.append(A[np.argsort(d)[:k]])
    return np.concatenate(out, axis=0)

GC = 127.3          # 故意放在非网格对齐的位置，打破对称性
print(f"{'GT 尺寸':>8s} {'候选均值':>10s} {'标准差':>9s} {'ATSS 阈值':>11s} "
      f"{'正样本数':>9s} {'正样本平均IoU':>14s} {'固定0.5能选出':>14s}")
info = {}
for size in [8, 32, 128]:
    g = (GC, GC, float(size), float(size))
    cand = topk_per_level(g, k=9)
    thr, v = atss_threshold(g, cand, lambda a, b: iou(a, b))
    pos = v >= thr - 1e-12
    n_fixed = int((v >= 0.5).sum())
    info[size] = dict(mean=float(v.mean()), std=float(v.std()), thr=thr,
                      n_pos=int(pos.sum()), q=float(v[pos].mean()), n05=n_fixed)
    print(f'{size:>8d} {v.mean():>10.4f} {v.std():>9.4f} {thr:>11.4f} '
          f'{int(pos.sum()):>9d} {v[pos].mean():>14.4f} {n_fixed:>14d}')

assert info[8]['thr'] < 0.3, 'ATSS 的阈值对小目标自动降下来（固定 0.5 会一个都选不出）'
assert info[128]['thr'] > 2 * info[8]['thr'], '阈值随目标变大而升高'
assert info[8]['n05'] == 0, '固定 IoU@0.5 对 8px 目标一个候选都选不出'
assert info[128]['n05'] > 0
assert info[8]['q'] < 0.5 * info[128]['q'], '小目标的正样本**质量**远低于大目标'
print('\n✅ ATSS 的阈值从数据里长出来，对小目标自动降低 —— 确实比固定 0.5 公平：')
print(f"   8px 目标在固定 IoU@0.5 下能选出 {info[8]['n05']} 个候选，"
      f"ATSS 把阈值降到 {info[8]['thr']:.3f} 后选出 {info[8]['n_pos']} 个。")
print(f"⚠️  但正样本的**质量**没有跟上：8px 的正样本平均 IoU 只有 {info[8]['q']:.3f}，"
      f"而 128px 是 {info[128]['q']:.3f}。")
print('    模型被要求从一堆几乎不含目标的 anchor 上回归出准确的框。')
print('✅ 解法：把 ATSS 内部的 IoU 换成 NWD，两个机制叠加（自适应 + 公平的尺子）。')

In [ ]:
def dynamic_k(gt, anchors, metric, q=10):
    '''SimOTA: k = max(1, round(sum(top-q 候选的度量值)))'''
    v = metric(np.tile(np.asarray(gt, float), (len(anchors), 1)), np.asarray(anchors, float))
    topq = np.sort(v)[-q:]
    return max(1, int(round(float(topq.sum())))), float(topq.sum())

print(f"{'GT 尺寸':>8s} {'top10 IoU 和':>13s} {'k(IoU版)':>10s} "
      f"{'top10 NWD 和':>13s} {'k(NWD版)':>10s}")
ks = {}
for size in [8, 32, 128]:
    g = (128., 128., float(size), float(size))
    k_i, s_i = dynamic_k(g, anchors, lambda a,b: iou(a,b))
    k_n, s_n = dynamic_k(g, anchors, lambda a,b: nwd(a,b))
    ks[size] = (k_i, k_n)
    print(f'{size:>8d} {s_i:>13.2f} {k_i:>10d} {s_n:>13.2f} {k_n:>10d}')

assert ks[8][0] < ks[128][0], 'IoU 版的 dynamic-k 给小目标的正样本**更少**'
assert ks[8][1] > ks[8][0],   'NWD 版把小目标的 k 拉了回来'
assert ks[128][1] < ks[128][0], '但纯 NWD 版把**大目标**砍了 —— 这正是必须混合的理由'
print(f'\n⚠️  **SimOTA 对小目标的作用方向是反的**：IoU 版给 8px 目标 k={ks[8][0]}，'
      f'给 128px 目标 k={ks[128][0]}。')
print('    本来正样本就稀缺的小目标，被 dynamic-k 又砍了一刀。')
print(f'✅ 换成 NWD 后 8px 的 k 从 {ks[8][0]} 回到 {ks[8][1]}。')
print(f'⚠️  但注意最后一行：**纯 NWD 把 128px 目标的 k 从 {ks[128][0]} 砍到了 {ks[128][1]}** ——')
print('    因为位置项是绝对距离，128px 目标的候选中心偏十几像素就被判成"很远"。')
print('    这就是本课反复强调的那条：**纯 NWD 会伤大目标，必须混合**')
print('    metric = (1-α)·IoU + α·NWD（α 按小目标占比设，TSR 上 0.5~0.8），')
print('    或按尺寸切换（sqrt(wh) < 32 用 NWD，其余用 IoU）。')
print('   一般判断：**任何以 IoU 为内部度量的自适应机制，都会继承 IoU 的尺度不公平；')
print('   而换成纯 NWD 又会把不公平倒向另一头。**')

# 验证混合度量能同时照顾两头
def mixed(a, b, alpha=0.7):
    return (1 - alpha) * iou(a, b) + alpha * nwd(a, b)
k_mix_8,   _ = dynamic_k((128., 128., 8., 8.),     anchors, mixed)
k_mix_128, _ = dynamic_k((128., 128., 128., 128.), anchors, mixed)
print(f'\n混合度量(α=0.7): 8px -> k={k_mix_8}, 128px -> k={k_mix_128}')
assert k_mix_8 > ks[8][0] and k_mix_128 >= ks[128][1], (k_mix_8, k_mix_128)
print('✅ 混合后两头都不塌：小目标的 k 被拉起来，大目标的 k 不至于掉到 1。')

## 6 · RFLA：KLD 的方向决定了公平性

In [ ]:
def kld_gauss(p, g):
    '''D_KL(N_p || N_g)，对角高斯，box = (cx,cy,w,h)。
       = 0.5[ tr(Σ_g⁻¹Σ_p) + (μ_g-μ_p)ᵀΣ_g⁻¹(μ_g-μ_p) - 2 + ln(|Σ_g|/|Σ_p|) ]
       **注意第二项是马氏距离：用「后一个参数」的协方差归一化。**'''
    mu_p, s_p = box_to_gauss(p); mu_g, s_g = box_to_gauss(g)
    var_p, var_g = s_p**2, s_g**2
    tr   = (var_p / var_g).sum(-1)
    maha = (((mu_g - mu_p) ** 2) / var_g).sum(-1)
    logd = (np.log(var_g) - np.log(var_p)).sum(-1)
    return 0.5 * (tr + maha - 2.0 + logd)

assert abs(float(kld_gauss((5,5,20,20), (5,5,20,20)))) < 1e-12, 'KL(x||x) = 0'
assert float(kld_gauss((0,0,8,8), (2,2,8,8))) > 0

# 方向 A：以 GT 为参考（位置项 ∝ 1/w²，比 IoU 还不公平）
print('方向 A：D_KL(pred ‖ GT)  —— 用 **GT 尺寸** 归一化位置项')
print(f"{'框边长':>8s} {'KLD(位移2px)':>14s} {'相对 8px 的比值':>16s}")
base_a = None
for w in [8, 16, 32, 64]:
    v = float(kld_gauss((2,2,w,w), (0,0,w,w)))
    base_a = base_a or v
    print(f'{w:>8d} {v:>14.6f} {v/base_a:>16.4f}')
v8  = float(kld_gauss((2,2,8,8),   (0,0,8,8)))
v64 = float(kld_gauss((2,2,64,64), (0,0,64,64)))
assert abs(v8 / v64 - 64.0) < 1e-6, (v8, v64)
print(f'⚠️  8px 的惩罚是 64px 的 {v8/v64:.0f} 倍 = (64/8)² —— **比 IoU 还不公平**')

# 方向 B（RFLA 用的）：以**有效感受野**为参考（σ 由网络层决定，与 GT 尺寸无关）
def erf_gauss(cx, cy, stride, erf_cells=2.0):
    '''把特征点的有效感受野建模成高斯：sigma_ERF = erf_cells * stride。
       用 box 表示则 w = h = 2*sigma。'''
    s = erf_cells * stride
    return (cx, cy, 2*s, 2*s)

print('\n方向 B：D_KL(GT ‖ ERF)  —— 用 **感受野尺寸** 归一化位置项（RFLA 的做法）')
print(f"{'GT 边长':>8s} {'KLD 总值':>12s} {'其中位置项(马氏)':>18s}")
maha_vals = []
for w in [8, 16, 32, 64]:
    e = erf_gauss(2.0, 2.0, stride=8, erf_cells=2.0)      # ERF 中心偏 GT 中心 2px
    v = float(kld_gauss((0., 0., float(w), float(w)), e))
    _, s_e = box_to_gauss(np.array(e, float))
    m = float((((np.array(e[:2]) - np.array([0., 0.]))**2) / (s_e**2)).sum())
    maha_vals.append(m)
    print(f'{w:>8d} {v:>12.4f} {m:>18.6f}')
assert max(maha_vals) - min(maha_vals) < 1e-12, '位置项与 GT 尺寸无关 —— 这才是公平性的来源'
print('\n✅ **RFLA 的公平性不来自 KLD，而来自「拿感受野当参考分布」这个选择** ——')
print('   σ_ERF 由网络层决定，与目标大小无关，位置项于是变成绝对距离除以固定尺度。')
print('⚠️  方向搞反（用 GT 当参考）会比 IoU 更不公平。这是很好的面试追问点。')

In [ ]:
# RFLA 的另一半：HLA 的「保底 k」—— 保证每个 GT 至少拿到 k 个正样本
def assign_with_floor(gts, anchors, metric, thr=0.5, k_min=3):
    '''① 先取 metric >= thr 的；② 不足 k_min 个的，用 metric 最大的前 k_min 个补齐。'''
    gts, anchors = np.asarray(gts, float), np.asarray(anchors, float)
    out = []
    for g in gts:
        v = metric(np.tile(g, (len(anchors), 1)), anchors)
        idx = np.flatnonzero(v >= thr)
        if len(idx) < k_min:
            idx = np.argsort(v)[-k_min:]
        out.append(np.sort(idx))
    return out

# 多尺度 anchor（每个格点 3 档），格距 8
anc2 = np.array([[x+4., y+4., float(sz), float(sz)]
                 for x in range(0, 128, 8) for y in range(0, 128, 8)
                 for sz in (8, 16, 32)])
gts2 = np.array([[64., 64., 8., 8.], [64., 64., 32., 32.]])

n_iou_small = int((iou(np.tile(gts2[0], (len(anc2),1)), anc2) >= 0.5).sum())
n_nwd_small = int((nwd(np.tile(gts2[0], (len(anc2),1)), anc2) >= 0.5).sum())
print(f'8px GT: IoU@0.5 匹配到 {n_iou_small} 个 anchor, NWD@0.5 匹配到 {n_nwd_small} 个')
assert n_iou_small <= 2 and n_nwd_small >= 5, (n_iou_small, n_nwd_small)

res = assign_with_floor(gts2, anc2, lambda a,b: nwd(a,b), thr=0.5, k_min=3)
for g, r in zip(gts2, res):
    print(f'  GT {g[2]:.0f}x{g[3]:.0f}px -> {len(r)} 个正样本')
assert all(len(r) >= 3 for r in res)
print('\n✅ 「保底 k」解决的是「某些 GT 完全没有监督」这种极端情况。')
print('⚠️  没有保底时，这些 GT 的位置会被当作**负样本**训练 ——')
print('    模型被主动教育「这里没有目标」，比单纯漏掉它们更糟。')

## 7 · IoU 变体家族：GIoU / DIoU / CIoU / EIoU / SIoU

In [ ]:
def _enclose(a, b):
    A, B = to_xyxy(a), to_xyxy(b)
    x1 = np.minimum(A[...,0], B[...,0]); y1 = np.minimum(A[...,1], B[...,1])
    x2 = np.maximum(A[...,2], B[...,2]); y2 = np.maximum(A[...,3], B[...,3])
    return x1, y1, x2, y2

def giou(a, b):
    i = iou(a, b)
    x1, y1, x2, y2 = _enclose(a, b)
    Ac = np.maximum((x2-x1)*(y2-y1), 1e-12)
    A, B = to_xyxy(a), to_xyxy(b)
    ar_a = (A[...,2]-A[...,0])*(A[...,3]-A[...,1])
    ar_b = (B[...,2]-B[...,0])*(B[...,3]-B[...,1])
    xi1 = np.maximum(A[...,0],B[...,0]); yi1 = np.maximum(A[...,1],B[...,1])
    xi2 = np.minimum(A[...,2],B[...,2]); yi2 = np.minimum(A[...,3],B[...,3])
    inter = np.clip(xi2-xi1,0,None)*np.clip(yi2-yi1,0,None)
    union = ar_a + ar_b - inter
    return i - (Ac - union) / Ac

def diou(a, b):
    a_, b_ = np.asarray(a,float), np.asarray(b,float)
    rho2 = ((a_[...,0]-b_[...,0])**2 + (a_[...,1]-b_[...,1])**2)
    x1,y1,x2,y2 = _enclose(a,b)
    c2 = np.maximum((x2-x1)**2 + (y2-y1)**2, 1e-12)
    return iou(a,b) - rho2/c2

def _v_aspect(a, b):
    a_, b_ = np.asarray(a,float), np.asarray(b,float)
    return (4/np.pi**2) * (np.arctan(b_[...,2]/np.maximum(b_[...,3],1e-12)) -
                           np.arctan(a_[...,2]/np.maximum(a_[...,3],1e-12)))**2

def ciou(a, b):
    i, v = iou(a,b), _v_aspect(a,b)
    alpha = v / np.maximum((1 - i) + v, 1e-12)
    return diou(a,b) - alpha*v

def eiou(a, b):
    a_, b_ = np.asarray(a,float), np.asarray(b,float)
    x1,y1,x2,y2 = _enclose(a,b)
    cw2 = np.maximum((x2-x1)**2, 1e-12); ch2 = np.maximum((y2-y1)**2, 1e-12)
    return (diou(a,b) - (a_[...,2]-b_[...,2])**2/cw2 - (a_[...,3]-b_[...,3])**2/ch2)

def siou(a, b, theta=4.0, eps=1e-9):
    a_, b_ = np.asarray(a,float), np.asarray(b,float)
    dx, dy = b_[...,0]-a_[...,0], b_[...,1]-a_[...,1]
    sigma = np.maximum(np.sqrt(dx**2+dy**2), eps)
    x = np.clip(np.abs(dy)/sigma, 0.0, 1.0)
    Lam = 1 - 2*np.sin(np.arcsin(x) - np.pi/4)**2         # 角度代价
    gam = 2 - Lam
    x1,y1,x2,y2 = _enclose(a,b)
    Cw = np.maximum(x2-x1, eps); Ch = np.maximum(y2-y1, eps)
    Delta = (1-np.exp(-gam*(dx/Cw)**2)) + (1-np.exp(-gam*(dy/Ch)**2))
    ow = np.abs(a_[...,2]-b_[...,2])/np.maximum(np.maximum(a_[...,2],b_[...,2]), eps)
    oh = np.abs(a_[...,3]-b_[...,3])/np.maximum(np.maximum(a_[...,3],b_[...,3]), eps)
    Omega = (1-np.exp(-ow))**theta + (1-np.exp(-oh))**theta
    return iou(a,b) - (Delta + Omega)/2

same = (10., 10., 20., 20.)
for f, nm in [(iou,'IoU'),(giou,'GIoU'),(diou,'DIoU'),(ciou,'CIoU'),(eiou,'EIoU'),(siou,'SIoU')]:
    assert abs(float(f(same, same)) - 1.0) < 1e-9, nm
print('✅ 六个度量在完全重合时都等于 1（一致性校验通过）')

In [ ]:
# 事实 ①：GIoU 奖励「把预测框放大」，NWD 惩罚它
GT = (0., 0., 8., 8.)
print(f"{'预测框边长':>11s} {'IoU':>8s} {'GIoU':>9s} {'DIoU':>9s} {'NWD':>9s}")
gs, ns = [], []
for w in [8, 16, 24, 40]:
    pred = (30., 30., float(w), float(w))       # 中心偏 30px（不相交）
    g_, n_ = float(giou(pred, GT)), float(nwd(pred, GT))
    gs.append(g_); ns.append(n_)
    print(f'{w:>11d} {float(iou(pred,GT)):>8.4f} {g_:>9.4f} '
          f'{float(diou(pred,GT)):>9.4f} {n_:>9.4f}')

assert all(float(iou((30.,30.,float(w),float(w)), GT)) == 0.0 for w in [8,16,24,40])
assert gs[-1] > gs[0] + 0.4, 'GIoU 随预测框放大而**变好** -> 优化器会朝这个方向走'
assert ns[-1] < ns[0],       'NWD 随预测框放大而变差 -> 方向相反'
print(f'\n⚠️  预测框 8x8 -> 40x40: GIoU {gs[0]:.4f} -> {gs[-1]:.4f}（"变好了"）')
print(f'✅ 同样的变化: NWD {ns[0]:.4f} -> {ns[-1]:.4f}（变差）')
print('    Σ 项显式惩罚尺寸不匹配 —— **GIoU 奖励放大，NWD 惩罚放大，方向相反**。')
print('    对小目标，GIoU 的这个性质直接产生系统性偏大的框。')

In [ ]:
# 事实 ②：CIoU 的长宽比项梯度 ∝ 1/w （小框上大一个量级）
def dv_dw(w0, ar=1.25, eps=1e-6):
    '''在「GT 为 w0 x w0 方形、预测框为 (ar*w0) x w0」这一点上，
       固定 h，用有限差分求 ∂v/∂w。理论值 ∝ h/(w²+h²) ∝ 1/w0。'''
    gt, h = (0., 0., float(w0), float(w0)), float(w0)
    f = lambda ww: float(_v_aspect((0., 0., ww, h), gt))
    return (f(w0*ar + eps) - f(w0*ar - eps)) / (2*eps)

print(f"{'w':>6s} {'|∂v/∂w|':>12s} {'理论 ∝1/w（以 w=4 归一）':>26s}")
g4 = abs(dv_dw(4))
for w in [4, 8, 32, 128, 200]:
    gv = abs(dv_dw(w))
    print(f'{w:>6d} {gv:>12.6f} {g4 * 4.0/w:>26.6f}')
    assert abs(gv - g4*4.0/w) / (g4*4.0/w) < 0.02, w

r = abs(dv_dw(8)) / abs(dv_dw(200))
assert abs(r - 25.0) < 1.0, r
print(f'\n⚠️  8px 与 200px 的长宽比项梯度相差 {r:.0f} 倍（= 200/8，精确的 1/w 关系）')
print('    归一化坐标下（除以图宽 1920），8px 对应的 h/(w²+h²) 因子约 120 ——')
print('    **这就是 CIoU 原文说「容易梯度爆炸、实现里直接去掉 w²+h² 分母」的由来**。')

# 事实 ③：CIoU 的长宽比项会完全失效（比例对了就不惩罚，哪怕大 5 倍）
pred_big, gt_small = (0., 0., 40., 40.), (0., 0., 8., 8.)
v = float(_v_aspect(pred_big, gt_small))
assert abs(v) < 1e-15, '长宽比相同 => v ≡ 0'
c_, e_ = float(ciou(pred_big, gt_small)), float(eiou(pred_big, gt_small))
print(f'\n预测框 40x40 vs GT 8x8（中心重合、长宽比相同）:')
print(f'  v(长宽比项) = {v:.1e}  -> CIoU 退化成 IoU = {c_:.4f}')
print(f'  EIoU 分别惩罚 w 与 h  -> EIoU = {e_:.4f}')
assert abs(c_ - float(iou(pred_big, gt_small))) < 1e-9
assert e_ < c_ - 1.0, 'EIoU 给出强得多的信号'
print(f'✅ 差了 {c_ - e_:.2f} —— 这正是 EIoU 存在的理由。')

In [ ]:
# 归一化坐标下的 L1 会系统性低估小目标的误差
IMG_W = 1920.0
cases = [('8px 框宽度错 50%',   8.0, 0.50), ('200px 框宽度错 10%', 200.0, 0.10)]
l1s = []
for name, w, rel in cases:
    err_px = w * rel
    l1_norm = err_px / IMG_W
    l1s.append(l1_norm)
    print(f'{name:<22s} 绝对误差 {err_px:5.1f}px  归一化 L1 = {l1_norm:.5f}')
assert l1s[1] / l1s[0] > 4.5
print(f'\n⚠️  客观上明显更差的那个预测（8px 框错了一半），'
      f'损失只有另一个的 {l1s[0]/l1s[1]:.2f} 倍。')
print('✅ 三种修法：① 按框尺寸归一化的相对误差；② scale-aware 加权；')
print('   ③ 直接用 L = 1 − NWD —— 它的位置项是**绝对像素距离**，天然没有这个问题。')

# NMS 里的 IoU 同样不公平
print(f'\nNMS 场景：两个框中心相距 5px')
for w in [8, 200]:
    i_ = float(iou((0.,0.,float(w),float(w)), (5.,0.,float(w),float(w))))
    n_ = float(nwd((0.,0.,float(w),float(w)), (5.,0.,float(w),float(w))))
    print(f'  {w:>3d}px 框: IoU = {i_:.3f} ({"会被合并" if i_>0.5 else "**保留成两个目标**"}), '
          f'NWD = {n_:.3f}')
assert float(iou((0.,0.,8.,8.), (5.,0.,8.,8.))) < 0.5
assert float(iou((0.,0.,200.,200.), (5.,0.,200.,200.))) > 0.9
print('✅ 把 NMS 的 IoU 也换成 NWD 能显著减少小目标的重复框 —— 改动只有一行。')

## 8 · scale-aware 加权：最优 γ 是内点，不是越大越好

In [ ]:
# 一个可精确求解的模型：共享一个参数 theta，两组目标的最优 theta 不同（容量冲突）
N_S, N_L   = 800, 200        # 小目标多、大目标少（TSR 的真实比例）
X_S, X_L   = 0.1, 1.0        # 「梯度杠杆」：归一化坐标下小目标的损失/梯度本来就小
TH_S, TH_L = 1.0, 2.0        # 两组的最优参数不同 -> 共享检测头的容量冲突
S_S, S_L   = 0.1, 1.0        # 尺度（用于加权）

def solve(gamma):
    '''加权最小二乘的闭式解： theta* = Σ w x² θ* / Σ w x² '''
    w_s, w_l = S_S ** (-gamma), S_L ** (-gamma)
    num = N_S*w_s*X_S**2*TH_S + N_L*w_l*X_L**2*TH_L
    den = N_S*w_s*X_S**2      + N_L*w_l*X_L**2
    return num/den, w_s, w_l, den

gammas = np.linspace(0, 3, 31)
thetas = np.array([solve(g)[0] for g in gammas])
# 指标：**按尺寸分桶等权**（每个桶算一次，再平均）—— 这正是分桶 mAP 的形式
score = 0.5*((thetas - TH_S)**2 + (thetas - TH_L)**2)
best = int(np.argmin(score))

print(f"{'gamma':>7s} {'theta*':>9s} {'小目标误差':>11s} {'大目标误差':>11s} {'分桶指标':>10s}")
for i in range(0, 31, 5):
    print(f'{gammas[i]:>7.2f} {thetas[i]:>9.4f} {(thetas[i]-TH_S)**2:>11.4f} '
          f'{(thetas[i]-TH_L)**2:>11.4f} {score[i]:>10.4f}')
print(f'\n最优 gamma = {gammas[best]:.2f}, 指标 {score[best]:.4f}')

assert 0 < best < len(gammas)-1, '最优 gamma 是**内点**，不是端点'
assert score[best] < score[0],   '适度加权确实比不加权好'
assert score[-1] > score[0],     'gamma=3 时比**完全不加权还差**'
print(f'⚠️  不加权(γ=0): {score[0]:.4f} -> 最优(γ={gammas[best]:.1f}): {score[best]:.4f} '
      f'-> 过度(γ=3.0): {score[-1]:.4f}')
print('✅ 两股相反的力：① 修正梯度杠杆不平衡（变好） ② 容量冲突（变差）')
print('   => 最优 γ 是内点。实操建议 **γ ∈ [0.25, 0.5]**（本模型的最优值偏大是因为')
print('      杠杆差距被刻意设成了 10 倍；真实场景差距更小，最优 γ 也更小）。')

In [ ]:
# 第二股反向力：噪声放大。小目标标注噪声大（±2px 对 8px 框是 25% 相对误差）
SIG_S, SIG_L = 0.30, 0.05

def theta_variance(gamma):
    '''加权最小二乘估计量的方差： Var = Σ w² x² σ² / (Σ w x²)²'''
    _, w_s, w_l, den = solve(gamma)
    num = N_S*w_s**2*X_S**2*SIG_S**2 + N_L*w_l**2*X_L**2*SIG_L**2
    return num / den**2

var = np.array([theta_variance(g) for g in gammas])
print(f"{'gamma':>7s} {'估计量方差':>14s} {'相对 γ=0':>12s}")
for i in range(0, 31, 5):
    print(f'{gammas[i]:>7.2f} {var[i]:>14.3e} {var[i]/var[0]:>12.1f}x')
assert np.all(np.diff(var) > 0), '方差随 gamma 单调递增'
assert var[-1] / var[0] > 100
print(f'\n⚠️  γ 从 0 到 3，估计量方差涨了 {var[-1]/var[0]:.0f} 倍 ——')
print('    这就是「加了小目标权重后训练开始不稳、多种子方差变大」的机制。')
print('✅ 三条实操约束：')
print('   ① γ ∈ [0.25, 0.5]（不是 1，更不是 2）')
print('   ② **必须归一化使权重均值为 1** —— 否则总梯度尺度改变 = 偷偷改了学习率')
print('   ③ **只对回归损失加权，不对分类损失加权**（分类不平衡交给 Focal / 采样）')

In [ ]:
# Focal Loss 的 gamma 在小目标场景要往回调：它会最用力地去学「漏标的真实目标」
p_neg = rng.beta(1.2, 12.0, size=200_000)        # 负样本得分：多数很低，少数很高
print(f"{'focal γ_f':>10s} {'最难的 1% 负样本占总损失的比例':>32s}")
shares = []
for gf in [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]:
    fl = (p_neg ** gf) * (-np.log(np.clip(1 - p_neg, 1e-12, None)))
    share = float(np.sort(fl)[-2000:].sum() / fl.sum())
    shares.append(share)
    print(f'{gf:>10.1f} {share:>31.1%}')
assert all(shares[i] < shares[i+1] for i in range(len(shares)-1)), 'γ_f 越大越集中于极难负样本'
assert shares[-1] > 3 * shares[0]
print(f'\n⚠️  γ_f 从 0 到 3，最难的 1% 负样本从占 {shares[0]:.1%} 涨到 {shares[-1]:.1%} 的损失。')
print('    **在小目标场景，这批"最难负样本"里有相当比例是漏标的真实目标**')
print('    （标注员看不清 6px 的牌子）—— γ_f=2 时模型被最用力地训练去把真目标压成背景。')
print('✅ TSR 建议：γ_f 降到 1.0–1.5；α 从 0.25 提到 0.4–0.5（正样本占比只有 0.04%）；')
print('   **更根本的一招：把低于标注下限的区域标成 ignore**，从源头切断这条错误监督。')
print('   若用 Quality Focal / VariFocal，把 IoU 软标签换成 NWD ——')
print('   小目标的 IoU 软标签系统性偏低，会把模型训成对小目标永远不自信。')

## ✏️ 练习 1：从零实现 NWD

实现 `my_nwd(a, b, C)`：输入两个 `(cx, cy, w, h)`（支持 `(...,4)` 广播），返回 NWD。
**不许调用上面的 `nwd` / `w2_sq`**，从高斯建模开始自己写。

In [ ]:
def my_nwd(a, b, C=12.8):
    # TODO: ① mu = (cx, cy)，Sigma^{1/2} 的对角元 = (w/2, h/2)
    #       ② W2² = ||Δmu||² + ||ΔSigma^{1/2}||_F²
    #       ③ 返回 exp(-sqrt(W2²)/C)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(float(my_nwd((3,7,11,5), (3,7,11,5))) - 1.0) < 1e-15
assert abs(float(my_nwd((0,0,8,8), (2,2,8,8))) - math.exp(-math.sqrt(8)/12.8)) < 1e-12
assert abs(float(my_nwd((0,0,8,8), (0,0,16,16))) - math.exp(-math.sqrt(32)/12.8)) < 1e-12
# 尺度不敏感（核心性质）
vals = [float(my_nwd((0,0,w,w), (2,2,w,w))) for w in [4,8,16,32,64,128,256]]
assert max(vals) - min(vals) < 1e-14, '相同绝对位移 -> 相同 NWD'
# 与参考实现逐元素对拍（含广播）
A = rng.uniform(0, 100, size=(50, 4)) + np.array([0,0,1,1])
B = rng.uniform(0, 100, size=(50, 4)) + np.array([0,0,1,1])
assert np.allclose(my_nwd(A, B, 12.8), nwd(A, B, 12.8), rtol=1e-12, atol=1e-14)
assert np.all(my_nwd(A, B) > 0), 'NWD 恒 > 0'
print(f'NWD((0,0,8,8),(2,2,8,8)) = {float(my_nwd((0,0,8,8),(2,2,8,8))):.6f}')
print(f'跨尺度极差 = {max(vals)-min(vals):.2e}')
print('✅ 练习 1 通过：三行实现，位置项是**绝对距离** —— 这就是 NWD 的全部')

## ✏️ 练习 2：尺度自适应阈值 + 容忍半径反解

实现 `scale_adaptive_thr(size, r0, tau_min, tau_max)`（让容忍半径恒为 `r0`）
和 `tolerance_radius(size, tau)`（给定阈值反算容忍半径，沿轴位移）。
两者应当互为逆运算（在未被 clip 的区间内）。

In [ ]:
def tolerance_radius(size, tau):
    # TODO: 解 (w-r)/(w+r) = tau
    raise NotImplementedError

def scale_adaptive_thr(size, r0=4.0, tau_min=0.15, tau_max=0.90):
    # TODO: 解 tolerance_radius(size, tau) = r0，再 clip
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert abs(tolerance_radius(8, 0.5) - 8/3) < 1e-12
assert abs(tolerance_radius(256, 0.3) - 256*0.7/1.3) < 1e-9
# 数值验证：把半径代回 IoU 应当恰好等于 tau
for w, t in [(8, 0.5), (32, 0.3), (128, 0.7)]:
    assert abs(float(iou((0,0,w,w), (tolerance_radius(w,t),0,w,w))) - t) < 1e-9
# 互逆性
for w in [8, 16, 32, 64]:
    assert abs(tolerance_radius(w, scale_adaptive_thr(w, r0=4.0)) - 4.0) < 1e-9, w
assert scale_adaptive_thr(4, r0=4.0) == 0.15, '4px 框需要 τ=0 才有 4px 半径 -> 被 tau_min 截断'
assert scale_adaptive_thr(1000, r0=4.0) == 0.90, '大框被 tau_max 截断（有意为之）'
# 单调性
ts = [scale_adaptive_thr(w) for w in [8, 16, 32, 64, 128]]
assert all(ts[i] <= ts[i+1] for i in range(len(ts)-1))
print(f"{'w':>6s} {'τ(w)':>8s} {'反算半径':>10s}")
for w in [8, 16, 32, 64, 256]:
    t = scale_adaptive_thr(w)
    print(f'{w:>6d} {t:>8.3f} {tolerance_radius(w, t):>10.2f}')
print('✅ 练习 2 通过：阈值表要**从设计目标反解**，不要拍脑袋定')

## ✏️ 练习 3：EIoU

实现 `my_eiou(pred, gt)`：
`EIoU = IoU − ρ²(中心)/c² − (w−w_g)²/c_w² − (h−h_g)²/c_h²`，
其中 c 是最小外接框的对角线长度、c_w / c_h 是外接框的宽和高。可以调用已有的 `iou` / `_enclose`。

In [ ]:
def my_eiou(pred, gt):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(float(my_eiou((10,10,20,20), (10,10,20,20))) - 1.0) < 1e-9, '完全重合 = 1'
# 与参考实现对拍
A = rng.uniform(0, 60, size=(60, 4)) + np.array([0,0,2,2])
B = rng.uniform(0, 60, size=(60, 4)) + np.array([0,0,2,2])
assert np.allclose(my_eiou(A, B), eiou(A, B), rtol=1e-9, atol=1e-12)
# 关键性质：长宽比相同但尺寸差 5 倍时，EIoU 远强于 CIoU
pb, gs = (0.,0.,40.,40.), (0.,0.,8.,8.)
assert abs(float(_v_aspect(pb, gs))) < 1e-15
assert float(my_eiou(pb, gs)) < float(ciou(pb, gs)) - 1.0
# 不相交时仍有梯度（值随距离单调下降）
vs = [float(my_eiou((d,0.,8.,8.), (0.,0.,8.,8.))) for d in [10, 20, 40, 80]]
assert all(vs[i] > vs[i+1] for i in range(len(vs)-1)), '不相交时仍单调 -> 有梯度'
print(f'EIoU(40x40 vs 8x8, 中心重合) = {float(my_eiou(pb, gs)):.4f}  '
      f'vs CIoU = {float(ciou(pb, gs)):.4f}')
print(f'不相交时随距离: {[round(v,4) for v in vs]}')
print('✅ 练习 3 通过：EIoU 修掉了 CIoU 的两个毛病（1/(w²+h²) 与「比例对了就不管」）')

## ✏️ 练习 4：scale-aware 权重（均值归一化）

实现 `scale_aware_weights(sizes, gamma, s_ref=None, w_max=None)`：
`w_i ∝ (s_ref/s_i)^gamma`，然后**归一化使权重均值恰为 1**（保持总梯度尺度不变）；
若给了 `w_max`，先把 `(s_ref/s_i)^gamma` 截断到 `w_max` 再归一化。
`s_ref` 缺省用 `sizes` 的中位数。

In [ ]:
def scale_aware_weights(sizes, gamma, s_ref=None, w_max=None):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
sz = np.array([8., 8., 16., 32., 64., 128.])
w0 = scale_aware_weights(sz, 0.0)
assert np.allclose(w0, 1.0), 'gamma=0 时所有权重都是 1'
w = scale_aware_weights(sz, 0.5)
assert abs(w.mean() - 1.0) < 1e-12, '**必须归一化使均值为 1**'
assert w[0] > w[-1], '小目标权重更大'
assert np.all(np.diff(w) <= 1e-12), '权重随尺寸单调不增'
wc = scale_aware_weights(sz, 2.0, w_max=4.0)
assert abs(wc.mean() - 1.0) < 1e-12
assert wc.max() / wc.min() <= 4.0 / (min(1.0, (np.median(sz)/sz.max())**2)) + 1e-9
# 截断确实生效：不截断时比值更大
wnc = scale_aware_weights(sz, 2.0)
assert wnc.max()/wnc.min() > wc.max()/wc.min()
print(f"{'尺寸':>7s} {'γ=0':>8s} {'γ=0.5':>8s} {'γ=1.0':>8s} {'γ=2.0(截断4)':>13s}")
w1 = scale_aware_weights(sz, 1.0)
for i, s in enumerate(sz):
    print(f'{s:>7.0f} {w0[i]:>8.3f} {w[i]:>8.3f} {w1[i]:>8.3f} {wc[i]:>13.3f}')
print('✅ 练习 4 通过：**归一化 + 截断** 是让 scale-aware 加权可控的两个必备开关')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_nwd(a, b, C=12.8):
    a_, b_ = np.asarray(a, float), np.asarray(b, float)
    mu_a, mu_b = a_[..., :2], b_[..., :2]           # 高斯均值 = 框中心
    s_a,  s_b  = a_[..., 2:] / 2.0, b_[..., 2:] / 2.0   # Sigma^{1/2} = diag(w/2, h/2)
    w2 = ((mu_a - mu_b) ** 2).sum(-1) + ((s_a - s_b) ** 2).sum(-1)
    return np.exp(-np.sqrt(w2) / C)

In [ ]:
# 练习 2 参考答案
def tolerance_radius(size, tau):
    return size * (1.0 - tau) / (1.0 + tau)

def scale_adaptive_thr(size, r0=4.0, tau_min=0.15, tau_max=0.90):
    return float(np.clip((size - r0) / (size + r0), tau_min, tau_max))

In [ ]:
# 练习 3 参考答案
def my_eiou(pred, gt):
    p, g = np.asarray(pred, float), np.asarray(gt, float)
    x1, y1, x2, y2 = _enclose(pred, gt)
    cw, ch = np.maximum(x2 - x1, 1e-12), np.maximum(y2 - y1, 1e-12)
    rho2 = (p[...,0]-g[...,0])**2 + (p[...,1]-g[...,1])**2
    return (iou(pred, gt) - rho2/(cw**2 + ch**2)
            - (p[...,2]-g[...,2])**2/cw**2 - (p[...,3]-g[...,3])**2/ch**2)

In [ ]:
# 练习 4 参考答案
def scale_aware_weights(sizes, gamma, s_ref=None, w_max=None):
    s = np.asarray(sizes, dtype=float)
    ref = float(np.median(s)) if s_ref is None else float(s_ref)
    w = (ref / np.maximum(s, 1e-12)) ** gamma
    if w_max is not None:
        w = np.minimum(w, float(w_max))
    return w / w.mean()                     # 归一化：保持总梯度尺度不变

---
## 🧪 真实工程胶囊：把 NWD 装进现有检测器

In [ ]:
RECIPE = r'''
# ============================================================================
# 小目标 · 分配与损失配置清单（按 ② -> ① -> ③ 的顺序做）
# ============================================================================

# ---- ② 换度量：NWD（改动最小、收益最确定）------------------------------------
import torch

def nwd(a, b, C=12.8, eps=1e-7):
    # a, b: (..., 4) 的 (cx, cy, w, h)，像素坐标。返回 (0, 1] 的相似度。
    mu = (a[..., :2] - b[..., :2]).pow(2).sum(-1)
    sg = ((a[..., 2:] - b[..., 2:]) / 2).pow(2).sum(-1)
    return torch.exp(-(mu + sg).clamp(min=eps).sqrt() / C)

# C 怎么定：取训练集所有 GT 的 sqrt(w*h) 的**均值**（不是中位数）
#   C = float(np.sqrt(gt_w * gt_h).mean())        # TSR 上典型 12~16
# tau 怎么定：由 r = -C*ln(tau) 反推
#   要求 r >= stride/2（网格量化） + 2px（标注抖动）
#   stride 8 -> r >= 6  ->  tau <= exp(-6/12.8) = 0.626   取 0.5~0.6

# ⚠️ 纯 NWD 会伤大目标（位置项是绝对距离，对大框区分度不足）。两种混合方式：
ALPHA = 0.7                                   # 按小目标占比设，TSR 上 0.5~0.8
metric = (1 - ALPHA) * iou(a, b) + ALPHA * nwd(a, b, C)         # ① 加权混合
# metric = torch.where(gt_size < 32, nwd(a,b,C), iou(a,b))      # ② 按尺寸切换

# 装进 MMDetection 的 assigner（把 IoUCalculator 换掉即可）
assigner = dict(type='MaxIoUAssigner', iou_calculator=dict(type='NWDCalculator', C=12.8),
                pos_iou_thr=0.5, neg_iou_thr=0.4, min_pos_iou=0.3)
# ATSS / SimOTA 同理：只换内部的 iou_calculator，自适应逻辑一行不改
# ⚠️ SimOTA 还要给 dynamic-k 设下界： k = max(3, round(topq.sum()))

# 回归损失也换（处处可导，不相交仍有梯度）
loss_bbox = dict(type='NWDLoss', C=12.8, loss_weight=2.0)   # L = 1 - NWD
# 或保守做法： loss = 0.5*(1-NWD) + 0.5*EIoULoss

# NMS 里的 IoU 同样不公平 —— 两个 8px 框相距 5px，IoU 只有 0.1 会被当两个目标
nms = dict(type='nwd_nms', C=12.8, iou_threshold=0.55)      # 显著减少小目标重复框

# ---- ① 改分配：保底 k（RFLA 的 HLA 思想，两行）-------------------------------
#   pos = (metric >= tau).nonzero()
#   if pos.numel() < K_MIN: pos = metric.topk(K_MIN).indices     # 保底
# 目的：避免某些 GT 完全没有正样本 —— 它们的位置会被当**负样本**训练，
#       模型被主动教育「这里没有目标」，比单纯漏掉更糟。

# ---- ③ 调损失：scale-aware 加权 + Focal 参数回调 ------------------------------
GAMMA_SCALE = 0.35                     # ∈ [0.25, 0.5]，不是 1 更不是 2
w = (s_ref / gt_size).clamp(max=4.0) ** GAMMA_SCALE
w = w / w.mean()                       # ← **必须归一化**，否则等于偷偷改了学习率
loss_reg = (w * per_sample_reg_loss).mean()     # 只加权**回归**，不加权分类

loss_cls = dict(type='FocalLoss', gamma=1.5, alpha=0.4)   # 从 (2.0, 0.25) 回调
# ⚠️ 小目标场景最"难"的负样本往往是**漏标的真实目标** ->
#    gamma=2 会最用力地训练模型把真目标压成背景。
# ✅ 更根本：把低于标注下限（如 <8px）的区域标成 ignore（既不算正也不算负）

# ---- 验证清单（每一项都要在合成数据上先跑通再上真数据）------------------------
#   [ ] nwd(x, x) == 1.0 ；nwd 恒 > 0
#   [ ] 相同绝对位移下，不同尺寸框的 NWD **完全相同**（这是恒等式）
#   [ ] 8px GT vs 32px anchor 中心重合时 NWD < 0.5（Σ 项没写漏）
#   [ ] 换 NWD 前后，统计每个 GT 的正样本数直方图 —— 小目标那一端应明显抬升
#   [ ] **分尺寸桶评测**：AP_[0,8) / AP_[8,16) / AP_[16,32) / AP_[32,96) / AP_96+
#       只看总 mAP 的话，"AP_s +4 / AP_l −1" 会显示成 +0.3（看起来像噪声）
#   [ ] 大目标是否掉点？掉了就调 ALPHA 或改成按尺寸切换

# ---- 反模式清单 ---------------------------------------------------------------
#   ✗ 直接把 IoU 阈值从 0.5 降到 0.3（8px 够了，但 256px 框的容忍半径变成 138px）
#   ✗ 照搬 AI-TOD 的纯 NWD 配置到有大目标的数据集（那个数据集没有大目标）
#   ✗ 用未修正的 CIoU（1/(w²+h²) 在归一化坐标下会梯度爆炸）
#   ✗ scale-aware 权重不归一化（总梯度尺度改变 = 偷偷改学习率）
#   ✗ 分类和回归同时加 scale 权重（与 Focal 的隐式加权叠加 = 过度）
#   ✗ 只改损失不改分配（第③层是在已有正样本集合上微调，度量错了它救不回来）
'''
print(RECIPE)
for key in ['def nwd', 'ALPHA', 'k = max(3', 'NWDLoss', 'nwd_nms', 'K_MIN',
            'w / w.mean()', 'gamma=1.5', 'ignore', '分尺寸桶评测', '反模式']:
    assert key in RECIPE, key
print('✅ 配方覆盖：换度量 -> 混合防大目标掉点 -> 保底 k -> 加权与 Focal 回调 -> 分桶验证')

### 小结

- **IoU 是相对度量，小目标面对的威胁是绝对像素量**（网格量化 ±4 px、标注抖动 ±2 px）。
  用相对度量衡量绝对误差，就是整个问题的根源。IoU 的位移灵敏度 = **4/w**。
- **NWD 三行就能实现**：框 → 2D 高斯（Σ = diag(w²/4, h²/4)）→ W₂² = 把框写成
  `[cx, cy, w/2, h/2]` 后的欧氏距离平方 → `exp(−√W₂²/C)`。
  它的灵敏度恒为 **√2/C**，与框大小**完全无关**（是恒等式，不是近似）。
- **三个关键性质**：① 相同绝对位移 → 相同相似度；② 不相交仍有梯度，且
  **惩罚放大预测框**（GIoU 恰好相反，它奖励放大）；③ 处处光滑、等值面是正圆。
- **决定性数字**：stride 8 网格上，即使 anchor 尺寸完美匹配，
  **8 px 目标在 IoU@0.5 下只有 ~25% 能匹配上**（16 px 85%，32 px 100%）；NWD@0.5 下是 100%。
- **NWD 不是免费的**：位置项是绝对距离 ⇒ 对大目标区分度不足。
  **必须混合**（`(1−α)·IoU + α·NWD`，α = 0.5–0.8）或按尺寸切换。照搬 AI-TOD 的纯 NWD 会掉点。
- **RFLA 的公平性不来自 KLD，而来自「拿感受野当参考分布」**——KLD 方向搞反
  （用 GT 当参考）会让位置项 ∝ 1/w²，**比 IoU 还不公平**。它真正值得抄的是 **HLA 的保底 k**。
- **任何以 IoU 为内部度量的自适应机制都会继承 IoU 的偏见**：ATSS 在极小目标上退化成
  「top-k 全收但质量极差」；**SimOTA 的 dynamic-k 对小目标方向是反的**（8 px 给 k=2，128 px 给 k=7）。
  修法都是「把内部的 IoU 换成 NWD」。
- **IoU 变体选择**：避开纯 GIoU（奖励放大）与未修正的 CIoU
  （∂v/∂w ∝ 1/w，8 px 比 200 px 大 25 倍；且长宽比一致时 v ≡ 0，大 5 倍也不罚）；
  用 **EIoU** 或 **1 − NWD**。**归一化坐标下的 L1 会把小目标的损失打 5 折**。
- **scale-aware 加权的最优 γ 是内点**（本模型 1.4，实操建议 0.25–0.5）：
  修正梯度杠杆（变好）与容量冲突 + 噪声放大（变差）两股力相抗；γ=3 时方差涨 380 倍。
  **必须归一化使均值为 1，且只加权回归**。
- **优先级 ② → ① → ③**：先把尺子换对（改动最小、且分配器内部就在用它），
  再补分配的保底逻辑，最后才微调损失权重。**给出这个排序的理由，比排序本身更能说明水平。**

下一站：**模块 04 · 切片推理与高分辨率策略**——当分配和损失都做对了，
剩下的最后一招是在推理时**提高目标的相对尺寸**。